<a href="https://colab.research.google.com/github/banshitarout16/T5-Text-Summarizer/blob/main/summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import files

uploaded = files.upload()

Saving samsum-test.csv to samsum-test.csv
Saving samsum-train.csv to samsum-train.csv
Saving samsum-validation.csv to samsum-validation.csv


In [3]:
!pip install transformers
!pip install "transformers[torch]"

In [4]:
import pandas as pd
import re
import torch
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [5]:
train_data = pd.read_csv("samsum-train.csv")
validation_data = pd.read_csv("samsum-validation.csv")

In [6]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [7]:
validation_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [8]:
## Random Sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=500, random_state=42).reset_index(drop=True)
train_data.shape

(4000, 3)

In [9]:
validation_data.shape

(500, 3)

In [10]:
# data Preprocessing ( remove HTML tags, unnecessary space & \r\n lines)

def clean_data(text):
    text=re.sub(r"\r\n", " ", text)
    text=re.sub(r"\s+", " ", text)
    text=re.sub(r"<.*?>", " ", text)
    text = text.strip().lower()
    return text

# apply clean_data to train & val data

train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_data)
validation_data["summary"] = validation_data["summary"].apply(clean_data)

In [11]:
validation_data.head(10)

,id,dialogue,summary
0,13680857,"edd: wow, did you hear that they're transferri...",rose and edd will be transferred to a new depa...
1,13716124,"tom: where is the ""sala del capitolo"" kevin: i...","""sala del capitolo"" tom is looking for is in t..."
2,13864418,patricia: the rowing practice is cancelled! ka...,the rowing practice is cancelled. a few member...
3,13729340,"tom: u ok? alex: yeah, pretty good. u? tom: ...",tom and alex had fun last night. they drank a ...
4,13818813,"patricia: hello, here's the fair-trade brand i...",patricia recommends a fair-trade brand she tal...
5,13729102,mark: what time is the breakfast? susanne: 8-1...,susanne will have breakfast at 8 am.
6,13680722,"george: ben, are you going to our choir rehear...",ben is sick and won't come to the choir rehear...
7,13730875,derek: hey derek: yo?? derek: ??? danny: let m...,danny would like to be left to sleep.
8,13730187,iza: monica: omg monica: yesssssss!!! iza: i...,iza has good news.
9,13865465,alice: did you know that amy had an abortion? ...,amy had an abortion.


### *Removed all the HTML tags, unnecessary space & \r\n lines*

In [12]:
# Embedding(tokenizers)

tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [13]:
#raw data -> tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True) #input value
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True) #Output value

    inputs["labels"] = targets["input_ids"] # token ids- add to input as labels
    return inputs

In [14]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
validation_dataset = validation_data.apply(tokenize, axis=1).tolist()

In [15]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [16]:
len(train_dataset[0]["input_ids"])

512

---
## headup:-
total token = 512

1 => EOs (end of sequals)

0 => padding

above embedded data has 3 major things:-
- input_ids = dialogues converted into token ids
- attention_mask = it shows where the [validid=1, invalid id=0] (ps-if there is token it shows->1, if no token(padding) it refelects->0
- labels = target(summary converted into token )

In [17]:
# working with Model

model = T5ForConditionalGeneration.from_pretrained("t5-small") #use for generational task

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [18]:
if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")

print("device:", device)
model.to(device)

device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [19]:
# defining training args

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay=0.01,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
)

In [20]:
# defining Trainer - the trainer class provides an API for features training in PyTorch

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset
)

In [21]:
# training

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.645737,0.381614
2,0.397080,0.360375
3,0.373668,0.354768
4,0.361432,0.350594
5,0.355313,0.349356
6,0.351061,0.348756


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.914048355102539, metrics={'train_runtime': 1274.4411, 'train_samples_per_second': 18.832, 'train_steps_per_second': 2.354, 'total_flos': 3248203235328000.0, 'train_loss': 0.914048355102539, 'epoch': 6.0})

In [22]:
# model load -> finetune -> model save

In [24]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [26]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]